# Indian address NER: fine-tune ModernBERT and TinyBERT

First choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
The last cell saves the models to Google Drive, because Colab deletes files when the session ends.

In [ ]:
!nvidia-smi -L
# If the repo is private, use: https://<GITHUB_TOKEN>@github.com/rohannayak032/pii-redaction-pipeline.git
!git clone -b detection-indian-address https://github.com/rohannayak032/pii-redaction-pipeline.git
%cd pii-redaction-pipeline
# pandas and pyarrow come with Colab; upgrading them breaks google-colab
!pip install -q -U transformers gliner
import os; os.environ["PYTHONPATH"] = "src"

## Data
Downloads gagan1985/indian-addresses-gold and builds `data/indian_address/{train,dev,test}.jsonl`. If `hf` is not found, use `huggingface-cli download` with the same arguments.

In [ ]:
!hf download gagan1985/indian-addresses-gold --repo-type dataset --local-dir data/indian-addresses-gold
!python -m pii_pipeline.detection.address_data

## Train
The number at the end is the epoch count. The epoch with the best dev score is saved to `data/models/<model>-ft`.

In [ ]:
!python -m pii_pipeline.detection.address_train modernbert 3

In [ ]:
!python -m pii_pipeline.detection.address_train tinybert 3

## Evaluate on the test set
Compares the base models with the fine-tuned ones. Latency and memory here are GPU numbers, so don't compare them with local CPU runs.

In [ ]:
for m in ["modernbert", "tinybert",
          "data/models/open-modernbert-indian-address-ner-ft",
          "data/models/open-tinybert-indian-address-ner-ft"]:
    !python -m pii_pipeline.detection.address_eval {m} test

## Save the models to Google Drive
Copy the folders into your local `data/models/` to use them.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pii-models && cp -r data/models/*-ft /content/drive/MyDrive/pii-models/